In [7]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks, select_period

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"

windows = {
    "train": (
        pd.Timestamp("2022-09-03"),
        pd.Timestamp("2022-09-07"),
    ),
    "validation": (
        pd.Timestamp("2022-09-07"),
        pd.Timestamp("2022-09-09"),
    ),
}

In [8]:
summaries = []

chunks = iter_transaction_chunks(
    csv_path,
    usecols=["timestamp", "payment_format", "is_laundering"],
)

for chunk in chunks:
    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    for split_name, (start, end) in windows.items():
        selected = select_period(chunk, start=start, end=end)

        if selected.empty:
            continue

        if not selected["is_laundering"].isin([0, 1]).all():
            raise ValueError("Expected laundering labels to be 0 or 1.")

        summary = (
            selected.groupby("payment_format", dropna=False)
            .agg(
                transactions=("is_laundering", "size"),
                laundering_transactions=("is_laundering", "sum"),
            )
            .reset_index()
        )
        summary["split"] = split_name
        summaries.append(summary)

if not summaries:
    raise ValueError("No transactions were found in the selected windows.")

label_counts = (
    pd.concat(summaries, ignore_index=True)
    .groupby(["split", "payment_format"], dropna=False)[
        ["transactions", "laundering_transactions"]
    ]
    .sum()
    .sort_index()
)

label_counts["laundering_rate_pct"] = (
    100
    * label_counts["laundering_transactions"]
    / label_counts["transactions"]
)

display(label_counts.round({"laundering_rate_pct": 3}))

transactions  laundering_transactions  \
split      payment_format                                          
train      ACH                   175504                     1583   
           Bitcoin                40508                       24   
           Cash                  143893                       29   
           Cheque                555975                       97   
           Credit Card           412488                       67   
           Wire                   51183                        0   
validation ACH                   116910                      890   
           Bitcoin                28643                       12   
           Cash                  103357                       25   
           Cheque                397108                       59   
           Credit Card           282817                       50   
           Wire                   36689                        0   

                           laundering_rate_pct  
split      payment_format                       
train      ACH                           0.902  
           Bitcoin                       0.059  
           Cash                          0.020  
           Cheque                        0.017  
           Credit Card                   0.016  
           Wire                          0.000  
validation ACH                           0.761  
           Bitcoin                       0.042  
           Cash                          0.024  
           Cheque                        0.015  
           Credit Card                   0.018  
           Wire                          0.000

In [9]:
label_counts.to_csv(
    project_root / "reports/lab_02_label_counts.csv"
)

### Interpretation of the label distribution

Both training and validation contain both classes. ACH accounts for
most laundering examples in both periods.

Wire has no positive examples in either period, so laundering recall
cannot be evaluated for wire.

Bitcoin has only 12 validation positives. One additional detection
changes its recall by 8.3 percentage points, so its recall estimate
should be interpreted cautiously.

In [10]:
selected_columns = [
    "timestamp",
    "amount_paid",
    "payment_currency",
    "amount_received",
    "receiving_currency",
    "payment_format",
    "is_laundering",
]

categorical_columns = [
    "payment_currency",
    "receiving_currency",
    "payment_format",
]

parts = {name: [] for name in windows}
row_offset = 0

chunks = iter_transaction_chunks(
    csv_path,
    usecols=selected_columns,
)

for chunk in chunks:
    # Preserve each transaction's position in the original CSV.
    chunk["source_row"] = range(
        row_offset,
        row_offset + len(chunk),
    )
    row_offset += len(chunk)

    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    for split_name, (start, end) in windows.items():
        selected = select_period(chunk, start=start, end=end)

        if not selected.empty:
            parts[split_name].append(selected)

In [11]:
datasets = {}

for split_name in windows:
    frame = pd.concat(
        parts.pop(split_name),
        ignore_index=True,
    )
    
    for column in categorical_columns:
        frame[column] = frame[column].astype("category")

    if not frame["is_laundering"].isin([0, 1]).all():
        raise ValueError("Expected laundering labels to be 0 or 1.")

    frame["is_laundering"] = frame["is_laundering"].astype("int8")

    datasets[split_name] = (
        frame
        .sort_values(["timestamp", "source_row"])
        .reset_index(drop=True)
    )

train = datasets["train"]
validation = datasets["validation"]

In [12]:
checks = []

for split_name, frame in datasets.items():
    expected = label_counts.loc[split_name]

    assert len(frame) == int(expected["transactions"].sum())
    assert int(frame["is_laundering"].sum()) == int(
        expected["laundering_transactions"].sum()
    )
    assert frame["source_row"].is_unique
    assert frame["timestamp"].is_monotonic_increasing

    checks.append({
        "split": split_name,
        "transactions": len(frame),
        "laundering_transactions": int(frame["is_laundering"].sum()),
        "memory_mib": frame.memory_usage(deep=True).sum() / 1024**2,
    })

assert train["timestamp"].max() < validation["timestamp"].min()

assert not train["source_row"].isin(
    validation["source_row"]
).any()

display(pd.DataFrame(checks).set_index("split").round(1))

,transactions,laundering_transactions,memory_mib
split,,,
train,1379551,1800,47.4
validation,965524,1036,33.1


In [13]:
display(pd.DataFrame({
    "train_missing": train.isna().sum(),
    "validation_missing": validation.isna().sum(),
}))

,train_missing,validation_missing
timestamp,0,0
amount_received,0,0
receiving_currency,0,0
amount_paid,0,0
payment_currency,0,0
payment_format,0,0
is_laundering,0,0
source_row,0,0


In [14]:
output_dir = project_root / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

train.to_parquet(
    output_dir / "lab02_train.parquet",
    index=False,
)
validation.to_parquet(
    output_dir / "lab02_validation.parquet",
    index=False,
)

### Modeling-data preparation

We retained every transaction in the training and validation windows,
using a compact selection of transaction-level columns.

This preserves all observed positive examples and the original class
proportions within each period.

The first baseline will use transaction attributes only. Account-history
features will be evaluated in a later experiment.

`is_laundering` is the target, and `source_row` is traceability metadata.
Neither belongs in the model's input features.

In [15]:
from time import perf_counter

from threadpoolctl import threadpool_limits
from src.modeling import make_features, build_model

train = pd.read_parquet(
    project_root / "data/processed/lab02_train.parquet"
)
validation = pd.read_parquet(
    project_root / "data/processed/lab02_validation.parquet"
)

X_train = make_features(train)
X_validation = make_features(validation)

y_train = train["is_laundering"]
y_validation = validation["is_laundering"]

model = build_model()

started = perf_counter()

with threadpool_limits(limits=2):
    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_validation)[:, 1]

training_seconds = perf_counter() - started
print(f"Fit and prediction took {training_seconds:.1f} seconds.")

Fit and prediction took 3.5 seconds.


In [16]:
from src.modeling import CATEGORICAL_FEATURES
encoder = (
    model.named_steps["preprocess"]
    .named_transformers_["categorical"]
)

for column, categories in zip(
    CATEGORICAL_FEATURES,
    encoder.categories_,
):
    print(column, categories)

payment_currency ['Australian Dollar' 'Bitcoin' 'Brazil Real' 'Canadian Dollar' 'Euro'
 'Mexican Peso' 'Ruble' 'Rupee' 'Saudi Riyal' 'Shekel' 'Swiss Franc'
 'UK Pound' 'US Dollar' 'Yen' 'Yuan']
receiving_currency ['Australian Dollar' 'Bitcoin' 'Brazil Real' 'Canadian Dollar' 'Euro'
 'Mexican Peso' 'Ruble' 'Rupee' 'Saudi Riyal' 'Shekel' 'Swiss Franc'
 'UK Pound' 'US Dollar' 'Yen' 'Yuan']
payment_format ['ACH' 'Bitcoin' 'Cash' 'Cheque' 'Credit Card' 'Wire']


In [17]:
payment_rates = (
    train.groupby("payment_format", observed=True)["is_laundering"]
    .mean()
)

payment_scores = (
    validation["payment_format"]
    .astype("string")
    .map(payment_rates)
    .fillna(y_train.mean())
    .to_numpy(dtype=float)
)